# Transformer

Notebook de evaluación del modelo Transformer sobre series temporales de acciones.

La estructura sigue la idea del notebook de ARIMA: preparación de datos, métricas, pruebas sobre distintos horizontes, sensibilidad a hiperparámetros y evaluación sobre el panel general.

A diferencia de ARIMA, el Transformer recibe ventanas de features OHLCV estacionarias y aprende a predecir un **retorno logarítmico a `horizon` días**.

## 1. Dependencias e imports

In [ ]:
!pip install -q pandas numpy matplotlib seaborn torch requests pydantic_settings

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import requests
import time

# Raíz del proyecto: .../api_ml
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.transformer import TransformerTrainConfig, TransformerTrendModel
from src.lstm import build_features, make_windows, FEATURE_NAMES

## 2. Descarga y cache de datos

Usamos la misma fuente de datos que el notebook de ARIMA para que la comparación entre modelos sea consistente.

In [ ]:
def get_ticker_data(ticker):
    r = requests.post(f"https://data-colector.onrender.com/historical-data/{ticker}")
    r.raise_for_status()
    body = r.json()
    return pd.DataFrame(body["data"])

ticker_stats = {}

def ticker_data(ticker, use_cache=True):
    if use_cache and ticker in ticker_stats:
        return ticker_stats[ticker]
    ticker_stats[ticker] = get_ticker_data(ticker)
    return ticker_stats[ticker]

def prepare_ohlcv(df):
    df = df.copy()
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date")
    return df.reset_index(drop=True)

## 3. Métricas

Para mantener la comparación con ARIMA usamos RMSE y accuracy direccional. Además separamos los aciertos cuando el retorno real es positivo o negativo.

In [ ]:
def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

def mae(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.mean(np.abs(actual - predicted)))

def directional_accuracy(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.mean(np.sign(actual) == np.sign(predicted)))

def up_accuracy(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mask = actual > 0
    return float(np.mean(predicted[mask] > 0)) if mask.any() else np.nan

def down_accuracy(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mask = actual < 0
    return float(np.mean(predicted[mask] < 0)) if mask.any() else np.nan

## 4. Predicción del retorno

El `TransformerTrendModel` ya hace la normalización y desnormalización del target internamente. Para evaluar de forma comparable y poder obtener una serie de predicciones, usamos directamente la misma red y los mismos `feature_mean`, `feature_std`, `target_mean` y `target_std` que utiliza `predict_df`.

In [ ]:
def predict_log_return(model, df):
    cfg = model.config
    feats, _ = build_features(df, cfg.horizon)

    if len(feats) < cfg.window:
        raise ValueError(f"No hay suficientes observaciones para window={cfg.window}")

    window = feats[-cfg.window:][None, :, :]
    x = torch.tensor(model._scale_features(window), dtype=torch.float32)

    model._net.eval()
    with torch.no_grad():
        scaled = float(model._net(x).item())

    return scaled * model.target_std + model.target_mean

def actual_log_return(df, start_idx, horizon):
    close = pd.to_numeric(df["close_amount"], errors="coerce")
    start = float(close.iloc[start_idx])
    end = float(close.iloc[start_idx + horizon])
    return float(np.log(end / start))

## 5. Evaluación rolling sobre un conjunto de test

Para cada ticker entrenamos el modelo **solo con los datos anteriores al período de test**. Después generamos predicciones desde cada fecha del período de test.

Esto es importante: no usamos las observaciones futuras para entrenar el Transformer. El modelo se entrena una vez y se lo evalúa sobre distintos puntos de entrada del período reservado.

In [ ]:
def evaluate_transformer(
    ticker,
    test_size=30,
    config=None,
    use_cache=True,
):
    df = prepare_ohlcv(ticker_data(ticker, use_cache=use_cache))

    if config is None:
        config = TransformerTrainConfig()

    horizon = config.horizon
    train_end = len(df) - test_size

    if train_end <= config.window + horizon:
        raise ValueError(f"Datos insuficientes para {ticker}")

    train_df = df.iloc[:train_end].copy()

    start = time.perf_counter()
    model = TransformerTrendModel(config=config)
    metrics_train = model.fit({ticker: train_df})
    duration = time.perf_counter() - start

    actual = []
    predicted = []
    dates = []

    first_origin = train_end
    last_origin = len(df) - horizon

    for origin in range(first_origin, last_origin):
        history = df.iloc[:origin].copy()
        pred = predict_log_return(model, history)
        real = actual_log_return(df, origin, horizon)

        actual.append(real)
        predicted.append(pred)

        if "date" in df.columns:
            dates.append(df["date"].iloc[origin])
        else:
            dates.append(origin)

    result = {
        "ticker": ticker,
        "model": model,
        "train_metrics": metrics_train,
        "actual": np.asarray(actual),
        "predicted": np.asarray(predicted),
        "dates": dates,
        "train_size": train_end,
        "test_size": len(actual),
        "duration": duration,
    }

    result["rmse"] = rmse(result["actual"], result["predicted"])
    result["mae"] = mae(result["actual"], result["predicted"])
    result["directional_accuracy"] = directional_accuracy(
        result["actual"], result["predicted"]
    )
    result["up_accuracy"] = up_accuracy(
        result["actual"], result["predicted"]
    )
    result["down_accuracy"] = down_accuracy(
        result["actual"], result["predicted"]
    )

    return result

## 6. Primera prueba

Empezamos con los mismos cinco tickers utilizados en las pruebas principales del notebook de ARIMA.

In [ ]:
tickers = ["BYMA", "YPFD", "SUPV", "GGAL", "BBAR"]

config = TransformerTrainConfig(
    window=30,
    horizon=5,
    d_model=32,
    nhead=4,
    num_layers=2,
    dim_feedforward=64,
    dropout=0.1,
    epochs=40,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=8,
    val_fraction=0.15,
    warmup_epochs=3,
    seed=42,
)

first_result = evaluate_transformer("BYMA", test_size=30, config=config)
{k: first_result[k] for k in [
    "ticker", "rmse", "mae", "directional_accuracy",
    "up_accuracy", "down_accuracy", "duration", "test_size"
]}

KeyError: 'close'

## 7. Visualización de una predicción

Acá vemos el retorno real a cinco días contra el retorno predicho por el Transformer para los distintos puntos del período de test.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(first_result["dates"], first_result["actual"], label="Real")
plt.plot(first_result["dates"], first_result["predicted"], label="Transformer")
plt.axhline(0, linewidth=1)
plt.title("Retorno logarítmico a 5 días - BYMA")
plt.xlabel("Fecha")
plt.ylabel("Log return")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. RMSE según cantidad de observaciones del período de test

En ARIMA se analiza el error al aumentar la cantidad de días predichos. En el Transformer el significado es ligeramente diferente: `horizon=5` permanece fijo y cambiamos cuántos puntos recientes utilizamos como período de evaluación.

Esto evita comparar artificialmente un modelo entrenado para retorno a 5 días con un forecast directo de ARIMA a 2, 5, 10, 15 y 30 días.

In [ ]:
ahead_time = [2, 5, 10, 15, 30]

rmse_results = []
mae_results = []

for ticker in tickers:
    ticker_rmse = []
    ticker_mae = []

    for test_size in ahead_time:
        result = evaluate_transformer(ticker, test_size=test_size, config=config)
        ticker_rmse.append(result["rmse"])
        ticker_mae.append(result["mae"])

    rmse_results.append(ticker_rmse)
    mae_results.append(ticker_mae)

rmse_results = np.asarray(rmse_results)
mae_results = np.asarray(mae_results)

rmse_results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(rmse_results)

ax.set_xticks(range(len(ahead_time)), labels=ahead_time)
ax.set_yticks(range(len(tickers)), labels=tickers)
ax.set_xlabel("Cantidad de observaciones del período de test")
ax.set_ylabel("Ticker")
ax.set_title("RMSE del Transformer")
fig.colorbar(im, ax=ax, label="RMSE")
plt.tight_layout()
plt.show()

## 9. Accuracy según período de test

In [ ]:
accuracy_results = []

for ticker in tickers:
    ticker_accuracy = []

    for test_size in ahead_time:
        result = evaluate_transformer(ticker, test_size=test_size, config=config)
        ticker_accuracy.append(result["directional_accuracy"])

    accuracy_results.append(ticker_accuracy)

accuracy_results = np.asarray(accuracy_results)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(accuracy_results, vmin=0, vmax=1)

ax.set_xticks(range(len(ahead_time)), labels=ahead_time)
ax.set_yticks(range(len(tickers)), labels=tickers)
ax.set_xlabel("Cantidad de observaciones del período de test")
ax.set_ylabel("Ticker")
ax.set_title("Accuracy direccional del Transformer")
fig.colorbar(im, ax=ax, label="Accuracy")
plt.tight_layout()
plt.show()

## 10. Sensibilidad al tamaño de ventana

El Transformer tiene un parámetro que ARIMA no tiene de la misma forma: `window`, es decir, cuántas observaciones históricas ve el modelo en cada predicción.

In [ ]:
windows = [20, 30, 40, 60]

window_results = []
window_durations = []

for ticker in tickers:
    ticker_results = []
    ticker_durations = []

    for window in windows:
        cfg = TransformerTrainConfig(
            window=window,
            horizon=5,
            d_model=32,
            nhead=4,
            num_layers=2,
            dim_feedforward=64,
            dropout=0.1,
            epochs=40,
            batch_size=64,
            learning_rate=1e-3,
            weight_decay=1e-4,
            patience=8,
            val_fraction=0.15,
            warmup_epochs=3,
            seed=42,
        )

        result = evaluate_transformer(ticker, test_size=30, config=cfg)
        ticker_results.append(result["rmse"])
        ticker_durations.append(result["duration"])

    window_results.append(ticker_results)
    window_durations.append(ticker_durations)

window_results = np.asarray(window_results)
window_durations = np.asarray(window_durations)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(window_results)

ax.set_xticks(range(len(windows)), labels=windows)
ax.set_yticks(range(len(tickers)), labels=tickers)
ax.set_xlabel("Window")
ax.set_ylabel("Ticker")
ax.set_title("RMSE según tamaño de ventana")
fig.colorbar(im, ax=ax, label="RMSE")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(window_durations)

ax.set_xticks(range(len(windows)), labels=windows)
ax.set_yticks(range(len(tickers)), labels=tickers)
ax.set_xlabel("Window")
ax.set_ylabel("Ticker")
ax.set_title("Tiempo de entrenamiento según tamaño de ventana")
fig.colorbar(im, ax=ax, label="Segundos")
plt.tight_layout()
plt.show()

## 11. Sensibilidad a la arquitectura

Probamos algunas arquitecturas pequeñas. Mantener modelos relativamente chicos es razonable para este problema porque la cantidad de observaciones financieras disponibles por ticker suele ser mucho menor que la utilizada para entrenar Transformers grandes.

In [ ]:
architectures = [
    {"d_model": 16, "nhead": 2, "num_layers": 1, "dim_feedforward": 32},
    {"d_model": 32, "nhead": 4, "num_layers": 2, "dim_feedforward": 64},
    {"d_model": 64, "nhead": 4, "num_layers": 2, "dim_feedforward": 128},
    {"d_model": 64, "nhead": 8, "num_layers": 3, "dim_feedforward": 128},
]

architecture_results = []

for arch in architectures:
    cfg = TransformerTrainConfig(
        window=30,
        horizon=5,
        dropout=0.1,
        epochs=40,
        batch_size=64,
        learning_rate=1e-3,
        weight_decay=1e-4,
        patience=8,
        val_fraction=0.15,
        warmup_epochs=3,
        seed=42,
        **arch,
    )

    values = []
    for ticker in tickers:
        result = evaluate_transformer(ticker, test_size=30, config=cfg)
        values.append(result["rmse"])

    architecture_results.append(values)

architecture_results = np.asarray(architecture_results)

architecture_labels = [
    "16/2/1",
    "32/4/2",
    "64/4/2",
    "64/8/3",
]

pd.DataFrame(
    architecture_results,
    index=architecture_labels,
    columns=tickers,
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(architecture_results)

ax.set_xticks(range(len(tickers)), labels=tickers)
ax.set_yticks(range(len(architecture_labels)), labels=architecture_labels)
ax.set_xlabel("Ticker")
ax.set_ylabel("d_model / heads / layers")
ax.set_title("RMSE según arquitectura")
fig.colorbar(im, ax=ax, label="RMSE")
plt.tight_layout()
plt.show()

## 12. Evaluación sobre el panel general

Usamos el mismo panel de acciones del notebook de ARIMA y resumimos RMSE, MAE y accuracy direccional por ticker.

In [ ]:
PANEL_LIDER = [
    "YPFD", "GGAL", "BMA", "TECO2", "PAMP", "BBAR", "TGSU2",
    "TXAR", "CEPU", "ALUA", "EDN", "BYMA", "LOMA", "SUPV",
    "CVH", "TRAN", "CRES", "HARG", "COME", "MIRG", "TGNO4",
    "VALO", "AGRO"
]

panel_results = []

for ticker in PANEL_LIDER:
    print(f"Evaluando {ticker}...")
    try:
        result = evaluate_transformer(ticker, test_size=30, config=config)
        panel_results.append({
            "ticker": ticker,
            "rmse": result["rmse"],
            "mae": result["mae"],
            "accuracy": result["directional_accuracy"],
            "up_accuracy": result["up_accuracy"],
            "down_accuracy": result["down_accuracy"],
            "duration": result["duration"],
            "test_samples": result["test_size"],
        })
    except Exception as exc:
        print(f"  Error en {ticker}: {exc}")

panel_df = pd.DataFrame(panel_results)
panel_df

## 13. Distribución de errores

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

axs[0].hist(panel_df["rmse"].dropna(), bins=8)
axs[0].set_title("Distribución de RMSE")
axs[0].set_xlabel("RMSE")
axs[0].set_ylabel("Cantidad de tickers")

axs[1].hist(panel_df["accuracy"].dropna(), bins=8, range=(0, 1))
axs[1].set_title("Distribución de Accuracy")
axs[1].set_xlabel("Accuracy direccional")
axs[1].set_ylabel("Cantidad de tickers")

plt.tight_layout()
plt.show()

## 14. Resultados promedio

In [ ]:
print(f"RMSE promedio: {panel_df['rmse'].mean():.6f}")
print(f"MAE promedio: {panel_df['mae'].mean():.6f}")
print(f"Accuracy promedio: {panel_df['accuracy'].mean():.4f}")
print(f"Up accuracy promedio: {panel_df['up_accuracy'].mean():.4f}")
print(f"Down accuracy promedio: {panel_df['down_accuracy'].mean():.4f}")

## 15. Comparación con un baseline simple

Antes de considerar que el Transformer funciona bien, conviene compararlo contra un modelo trivial. Para dirección, un baseline natural es predecir que el próximo retorno será cero.

Si el Transformer no supera claramente este baseline, mejorar la arquitectura probablemente no sea la solución: habría que revisar features, target o el esquema de validación.

In [ ]:
baseline_accuracy = []

for ticker in PANEL_LIDER:
    try:
        df = prepare_ohlcv(ticker_data(ticker))
        horizon = config.horizon
        test_size = 30
        train_end = len(df) - test_size

        actual = [
            actual_log_return(df, origin, horizon)
            for origin in range(train_end, len(df) - horizon)
        ]

        predicted = np.zeros(len(actual))

        baseline_accuracy.append({
            "ticker": ticker,
            "accuracy": directional_accuracy(actual, predicted),
            "rmse": rmse(actual, predicted),
        })
    except Exception:
        pass

baseline_df = pd.DataFrame(baseline_accuracy)
baseline_df

In [ ]:
comparison = panel_df[["ticker", "rmse", "accuracy"]].merge(
    baseline_df,
    on="ticker",
    suffixes=("_transformer", "_baseline"),
)

comparison["rmse_improvement"] = (
    comparison["rmse_baseline"] - comparison["rmse_transformer"]
)

comparison["accuracy_improvement"] = (
    comparison["accuracy_transformer"] - comparison["accuracy_baseline"]
)

comparison.sort_values("accuracy_improvement", ascending=False)

## 16. Conclusiones

Las conclusiones deberían basarse en los resultados obtenidos al ejecutar el notebook. En particular conviene mirar:

- si el Transformer supera al baseline;
- si el accuracy direccional se mantiene por encima de 50% de forma consistente;
- qué tamaño de ventana obtiene el mejor compromiso entre error y tiempo;
- si aumentar `d_model` o la cantidad de capas realmente mejora el test;
- si los resultados son consistentes entre tickers o dependen de unas pocas acciones.

No conviene elegir la arquitectura solamente por el mejor resultado de una única corrida: el modelo usa un `seed`, pero el comportamiento puede seguir siendo sensible a la muestra utilizada.